# MTBench_finance_stock — dataset exploration

Answers to 7 questions about [`GGLabYale/MTBench_finance_stock`](https://huggingface.co/datasets/GGLabYale/MTBench_finance_stock).

The dataset is **~16.5 GB** (≈100 parquet shards, 2813 tickers × ~191k bars × 10 fields), so we
**never load it all into memory**: `load_dataset(..., split="train")` returns a memory-mapped Arrow
table and individual rows are read lazily. Where a question only needs a sense of the data (e.g. price
ranges) we **randomly sample** tickers and timesteps instead of reading everything.

In [1]:
import datetime as dt
import numpy as np
from datasets import load_dataset

# Memory-mapped: this does NOT pull the 16.5 GB into RAM; rows are read on demand.
ds = load_dataset("GGLabYale/MTBench_finance_stock", split="train")
N_TICKERS = len(ds)
COLUMNS = ds.column_names
print(f"{N_TICKERS} tickers (rows)")
print("columns:", COLUMNS)


def get_sample(n, seed=0):
    # Deterministic random subset of ticker indices.
    n = min(n, N_TICKERS)
    return np.random.default_rng(seed).choice(N_TICKERS, size=n, replace=False)


def column(name):
    # Memory-mapped view exposing a single column for cheap per-row reads.
    return ds.select_columns([name])

/home/edan/miniconda3/envs/jax/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo card metadata block was not found. Setting CardData to empty.


2813 tickers (rows)
columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'transactions', 'otc', 'real_timestamp']


## 1. What do each of the fields mean?

These are **5-minute OHLCV aggregate bars** (Polygon.io-style) for US equities. Each **row is one ticker**;
each field is a list with **one value per 5-minute bar**.

| field | meaning |
|---|---|
| `timestamp` | Bar **start** time, Unix epoch **milliseconds** (UTC), snapped to a regular 5-min grid shared by all tickers. |
| `open` | First trade price in the bar. |
| `high` | Highest trade price in the bar. |
| `low` | Lowest trade price in the bar. |
| `close` | Last trade price in the bar. |
| `volume` | Total shares traded during the bar. |
| `vwap` | Volume-weighted average price during the bar. |
| `transactions` | Number of individual trades in the bar. |
| `otc` | Over-the-counter flag; **null in every sampled row** (all exchange-listed). |
| `real_timestamp` | The **actual** source-bar timestamp before alignment to the grid (see Q4). |

The cell below confirms the 5-minute spacing and that bars cover US regular trading hours
(first bar 14:30 UTC = 09:30 US/Eastern, the market open).

In [2]:
ex = ds[0]
print("Example values from ticker 0 (first 3 bars):")
for c in COLUMNS:
    print(f"  {c:15s}: {ex[c][:3]}")

ts = np.array(ds[0]["timestamp"])
diffs = np.diff(ts)
vals, counts = np.unique(diffs, return_counts=True)
mode = vals[np.argmax(counts)]
print()
print(f"Most common bar spacing: {mode} ms = {mode/1000/60:.0f} min "
      f"({counts.max()/len(diffs):.1%} of gaps; the rest are overnight/weekend gaps)")
print(f"Span: {dt.datetime.utcfromtimestamp(ts[0]/1000)} -> "
      f"{dt.datetime.utcfromtimestamp(ts[-1]/1000)} UTC")
print(f"First/last bar time-of-day (UTC): "
      f"{dt.datetime.utcfromtimestamp(ts[0]/1000).time()} .. "
      f"{dt.datetime.utcfromtimestamp(ts[-1]/1000).time()}  (= 09:30..15:55 US/Eastern)")

Example values from ticker 0 (first 3 bars):
  timestamp      : [1385389800000, 1385390100000, 1385390400000]
  open           : [54.11, 54.04, 53.94]
  high           : [54.13, 54.04, 53.96]
  low            : [54.09, 53.98, 53.94]
  close          : [54.09, 53.98, 53.95]
  volume         : [25266.0, 1100.0, 4702.0]
  vwap           : [54.11, 54.0082, 53.9468]
  transactions   : [4, 9, 37]
  otc            : [None, None, None]
  real_timestamp : [1385389800000, 1385390100000, 1385390400000]



Most common bar spacing: 300000 ms = 5 min (98.7% of gaps; the rest are overnight/weekend gaps)
Span: 2013-11-25 14:30:00 -> 2023-09-01 19:55:00 UTC
First/last bar time-of-day (UTC): 14:30:00 .. 19:55:00  (= 09:30..15:55 US/Eastern)


## 2. Do all tickers have the same number of samples?

**Yes.** Every sampled ticker has exactly **191,010** bars (sampling 100 tickers below).

In [3]:
samp = get_sample(100, seed=2)
tcol = column("timestamp")
lengths = np.array([len(tcol[int(i)]["timestamp"]) for i in samp])
uniq, cnt = np.unique(lengths, return_counts=True)
print(f"Sampled {len(samp)} tickers")
print("Distinct sample-count values found:", dict(zip(uniq.tolist(), cnt.tolist())))

Sampled 100 tickers
Distinct sample-count values found: {191010: 100}


## 3. Are the timesteps synced across tickers (is the nth sample the same instant for all)?

**Yes for `timestamp`.** The `timestamp` array is identical across tickers, so index `n` is the same
wall-clock instant for every ticker. **`real_timestamp` is *not* synced** — it holds each ticker's own
actual bar time (see Q4).

In [4]:
samp = get_sample(15, seed=3)
tcol = column("timestamp")
rcol = column("real_timestamp")

base_ts = np.array(tcol[int(samp[0])]["timestamp"])
ts_synced = all(np.array_equal(np.array(tcol[int(i)]["timestamp"]), base_ts) for i in samp[1:])

base_rt = np.array(rcol[int(samp[0])]["real_timestamp"])
rt_synced = all(np.array_equal(np.array(rcol[int(i)]["real_timestamp"]), base_rt) for i in samp[1:])

print(f"`timestamp` identical across {len(samp)} sampled tickers:      {ts_synced}")
print(f"`real_timestamp` identical across the same tickers: {rt_synced}")

`timestamp` identical across 15 sampled tickers:      True
`real_timestamp` identical across the same tickers: False


## 4. What is the difference between `timestamp` and `real_timestamp`?

`timestamp` is a **normalized grid** shared by all tickers (regular 5-min timeline). `real_timestamp` is
the **raw timestamp of the underlying bar** before it was snapped onto that grid. They're equal for most
bars but differ by a small offset (typically ±1–3 min, sometimes several) on a fraction of bars; that
fraction is larger for less-liquid tickers whose real bars don't land exactly on the grid.

In [5]:
samp = get_sample(8, seed=4)
both = ds.select_columns(["timestamp", "real_timestamp"])
all_delta = []
print("Per-ticker fraction where timestamp != real_timestamp:")
for i in samp:
    r = both[int(i)]
    t = np.array(r["timestamp"]); rt = np.array(r["real_timestamp"])
    d = t - rt
    differ = d != 0
    if differ.any():
        all_delta.append(d[differ])
        print(f"  ticker {int(i):5d}: {differ.mean():6.2%} differ | "
              f"median |offset| {np.median(np.abs(d[differ]))/1000:.0f}s")
    else:
        print(f"  ticker {int(i):5d}: none differ")

alld = np.concatenate(all_delta)
uo, co = np.unique(alld, return_counts=True)
print()
print("Most common (timestamp - real_timestamp) offsets over differing bars:")
for k in np.argsort(-co)[:6]:
    print(f"  {int(uo[k]/1000):+5d}s : {int(co[k])} bars")

i0 = int(samp[0]); r = both[i0]
t = np.array(r["timestamp"]); rt = np.array(r["real_timestamp"]); j = int(np.argmax(t != rt))
print()
print(f"Example (ticker {i0}, bar {j}):")
print("  timestamp      =", dt.datetime.utcfromtimestamp(t[j]/1000), "(snapped to 5-min grid)")
print("  real_timestamp =", dt.datetime.utcfromtimestamp(rt[j]/1000), "(actual source bar time)")

Per-ticker fraction where timestamp != real_timestamp:
  ticker  2038: 11.46% differ | median |offset| 60s


  ticker  2728: 88.84% differ | median |offset| 360s


  ticker   227: 83.18% differ | median |offset| 180s
  ticker  2474: 33.07% differ | median |offset| 60s


  ticker  2744: 86.64% differ | median |offset| 240s
  ticker  2647:  2.71% differ | median |offset| 60s


  ticker  2642: 32.02% differ | median |offset| 60s
  ticker  1436: 77.48% differ | median |offset| 240s

Most common (timestamp - real_timestamp) offsets over differing bars:
    +60s : 154439 bars
    -60s : 90392 bars
   +120s : 62399 bars
   -120s : 48907 bars
   +180s : 39065 bars
   -180s : 33929 bars



Example (ticker 2038, bar 1):
  timestamp      = 2013-11-25 14:35:00 (snapped to 5-min grid)
  real_timestamp = 2013-11-25 14:34:00 (actual source bar time)


## 5. What are the ranges of prices?

Sampling random (ticker, timestep) cells across the whole dataset rather than reading everything.
Prices span from **under \$1** to **~\$40,000** per share, median **~\$39**, with ~98% of sampled
values between ~\$2 and ~\$4,500. `open`/`high`/`low`/`close`/`vwap` all track each other closely,
and the range varies widely by ticker.

In [6]:
samp = get_sample(60, seed=5)
price_fields = ["open", "high", "low", "close", "vwap"]
pf = ds.select_columns(price_fields)
rng = np.random.default_rng(50)

buckets = {k: [] for k in price_fields}
for i in samp:
    r = pf[int(i)]
    ti = rng.choice(len(r["close"]), 300, replace=False)
    for k in price_fields:
        buckets[k].append(np.array(r[k], dtype=np.float64)[ti])

print(f"{len(samp)} tickers x 300 random timesteps = {len(samp)*300} samples per field")
print()
print(f"{'field':6s} {'min':>10s} {'p1':>9s} {'median':>9s} {'p99':>10s} {'max':>11s}")
for k in price_fields:
    a = np.concatenate(buckets[k]); a = a[~np.isnan(a)]
    print(f"{k:6s} {a.min():10.4f} {np.percentile(a,1):9.2f} {np.median(a):9.2f} "
          f"{np.percentile(a,99):10.1f} {a.max():11.1f}")

# How much the price range varies between tickers.
cl = column("close")
spans = np.array([(np.nanmin(c), np.nanmax(c))
                  for c in (np.array(cl[int(i)]["close"], float) for i in get_sample(40, seed=51))])
print()
print(f"Per-ticker close min: median {np.median(spans[:,0]):.2f}, "
      f"across tickers [{spans[:,0].min():.2f}, {spans[:,0].max():.2f}]")
print(f"Per-ticker close max: median {np.median(spans[:,1]):.2f}, "
      f"across tickers [{spans[:,1].min():.2f}, {spans[:,1].max():.2f}]")

60 tickers x 300 random timesteps = 18000 samples per field

field         min        p1    median        p99         max
open       0.8613      2.32     39.05     4508.2     39763.7
high       0.8650      2.32     39.08     4508.2     39763.7
low        0.8613      2.32     39.05     4508.2     39763.7
close      0.8650      2.32     39.06     4508.2     39763.7
vwap       0.8632      2.32     39.06     4508.2     39763.7



Per-ticker close min: median 14.71, across tickers [0.16, 66.45]
Per-ticker close max: median 61.09, across tickers [10.79, 559.06]


## 6. Will the open of timestep n always equal the close of timestep n-1?

**No.** They match only **~17%** of the time. Consecutive 5-min bars have small gaps (median ≈ \$0.03),
and equality is even rarer across session boundaries (overnight/weekend) than within a trading day.

In [7]:
samp = get_sample(40, seed=6)
oc = ds.select_columns(["open", "close"])
frac_eq, med_gap = [], []
for i in samp:
    r = oc[int(i)]
    o = np.array(r["open"], float); c = np.array(r["close"], float)
    frac_eq.append(np.nanmean(o[1:] == c[:-1]))
    med_gap.append(np.nanmedian(np.abs(o[1:] - c[:-1])))

print(f"Over {len(samp)} tickers:")
print(f"  mean fraction where open[n] == close[n-1]: {np.mean(frac_eq):.2%}")
print(f"  median |open[n] - close[n-1]|:             ${np.median(med_gap):.4f}")

# Intraday vs session-boundary gaps for one ticker.
i0 = int(samp[0]); r = oc[i0]
o = np.array(r["open"], float); c = np.array(r["close"], float)
t = np.array(column("timestamp")[i0]["timestamp"])
boundary = np.diff(t) > 300000          # gap > 5 min => overnight / weekend
eq = o[1:] == c[:-1]
print()
print(f"Ticker {i0}: open==prev_close  "
      f"intraday {np.nanmean(eq[~boundary]):.2%}  vs  across-session {np.nanmean(eq[boundary]):.2%}")

Over 40 tickers:
  mean fraction where open[n] == close[n-1]: 17.32%
  median |open[n] - close[n-1]|:             $0.0280

Ticker 1041: open==prev_close  intraday 6.15%  vs  across-session 1.22%


## 7. Are all tickers present for the full timespan?

**Yes, within the sample.** No NaN closes, no zero-volume opening bars, and no long leading run of
constant prices — every sampled ticker has genuine trading activity from the start (2013-11) to the
end (2023-09). The dataset looks **curated to names that traded across the entire ~10-year window**
(i.e. survivorship-selected); it is not padded for late-listed tickers.

In [8]:
samp = get_sample(50, seed=7)
sub = ds.select_columns(["close", "volume"])
lead_flat, zerovol_start, nan_any = [], 0, 0
n_bars = None
for i in samp:
    r = sub[int(i)]
    c = np.array(r["close"], float); v = np.array(r["volume"], float)
    n_bars = len(c)
    k = 1
    while k < len(c) and c[k] == c[0]:
        k += 1
    lead_flat.append(k)
    zerovol_start += int(v[0] == 0)
    nan_any += int(np.isnan(c).any())

print(f"Over {len(samp)} tickers (each {n_bars} bars, span 2013-11 -> 2023-09):")
print(f"  tickers with any NaN close:                       {nan_any}")
print(f"  tickers with zero volume in the first bar:        {zerovol_start}")
print(f"  longest leading run of constant close (any tk):   {max(lead_flat)} bars "
      f"(tiny => no back-fill before listing)")

Over 50 tickers (each 191010 bars, span 2013-11 -> 2023-09):
  tickers with any NaN close:                       0
  tickers with zero volume in the first bar:        0
  longest leading run of constant close (any tk):   7 bars (tiny => no back-fill before listing)


## Summary

1. **Fields** — 5-min OHLCV bars: `open/high/low/close` prices, `volume` (shares), `vwap`, `transactions` (trade count), `otc` flag (null here), and two timestamps (epoch ms, UTC).
2. **Same #samples?** — Yes, **191,010** bars per ticker.
3. **Synced timesteps?** — Yes for `timestamp` (identical grid across tickers); `real_timestamp` is not.
4. **`timestamp` vs `real_timestamp`** — `timestamp` is the normalized shared 5-min grid; `real_timestamp` is the raw bar time (off by ±1–5 min on some bars).
5. **Price range** — under \$1 to ~\$40,000, median ~\$39; varies a lot per ticker.
6. **open[n] == close[n-1]?** — No, only ~17% of the time (small gaps; even rarer across sessions).
7. **All tickers full timespan?** — Yes within the sample; dataset is survivorship-curated to the full 2013→2023 window.